# 04. 딜러별 계약출고 관리 — 월타겟 달성 예측용 데이터 탐색

**목표**: 딜러별 월 진행 중(D일 시점) 계약/출고 실적으로 **월말 타겟 달성 여부**를 예측하기 위해 필요한 원천 테이블을 실제로 연결·조회한다.

근거: `simon/ML_실행설계서_10개항목.md` ML① 항목 — 🟢 바로 가능 (일별 실적+타겟 패널, 딜러 21·모델 39, 272개월)

### 필요 테이블
| DB(엔드포인트) | 스키마.테이블 | 용도 |
|---|---|---|
| BP_KTWS → KPI_W | `dbo.OM_CONTRACT` | 딜러별 일별 계약 실적 (CONTRACT_DT, DEALER_ID, MODEL_CD, SFX_CD, CONTRACT_STAT_CD, SOLD_YN, ALLOCATION_YN 등) |
| BP_KTWS → KPI_W | `ktws.FCT_SALES_TARGET_DAILY` | 딜러·모델별 월 타겟 및 일별 타겟(ac_trgt, co_trgt) |
| BP_KTWS → KPI_W | `dbo.VS_SFX` / `dbo.VS_MODEL` | 사양(SFX)·모델 차원 테이블 |
| BP_KTWS → KPI_W | `ktws.DIM_CALENDAR` | 영업일 캘린더 (요일/영업일 보정용) |

> 참고: 과거 `VT_DAILY_SALES_TREND`는 갱신이 멈춘 legacy 테이블이므로 사용하지 않고, 실적은 `OM_CONTRACT`에서 직접 파생한다.

## 0. 준비

In [1]:
import pyodbc
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# BP/KTWS 엔드포인트 (KPI_W: OM_CONTRACT, FCT_SALES_TARGET_DAILY 등이 위치한 DB)
BPKTWS_SERVER = "6orm62c43rguff2mpdxwqy76tu-xhehnpiu2bautgglhximxpyqca.datawarehouse.fabric.microsoft.com"
DATABASE = "KPI_W"

def make_conn(server, database=None):
    db_part = f"DATABASE={database};" if database else ""
    conn_str = f"""
    DRIVER={{ODBC Driver 17 for SQL Server}};
    SERVER={server},1433;
    {db_part}
    Encrypt=yes;
    TrustServerCertificate=no;
    Authentication=ActiveDirectoryInteractive;
    Connection Timeout=30;
    """
    return pyodbc.connect(conn_str)

conn = make_conn(BPKTWS_SERVER, DATABASE)
pd.read_sql("SELECT DB_NAME() AS current_db", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2346834732.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("SELECT DB_NAME() AS current_db", conn)


,current_db
0,KPI_W


## 1. 딜러별 일별 계약 실적 — `dbo.OM_CONTRACT`

월말 달성 예측의 타깃(라벨) 및 실적 피처를 만드는 핵심 원천. 계약일 기준 최근 데이터를 확인한다.

In [2]:
om_contract = pd.read_sql("""
SELECT TOP 1000
    contract_dt,
    dealer_id,
    shop_cd,
    brand_cd,
    model_cd,
    variant_cd,
    sfx_cd,
    col_combi_cd,
    contract_stat_cd,
    sold_yn,
    allocation_yn,
    delivery_actual_dt
FROM dbo.OM_CONTRACT
ORDER BY contract_dt DESC
""", conn)

print(om_contract.shape)
om_contract.head(20)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\4076369220.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  om_contract = pd.read_sql("""


(1000, 12)


,contract_dt,dealer_id,shop_cd,brand_cd,model_cd,variant_cd,sfx_cd,col_combi_cd,contract_stat_cd,sold_yn,allocation_yn,delivery_actual_dt
0,2026-02-12,CW00000,CW00000,L,RX,AALH15L-AWXGBA,RJ,223-21,D4,Y,Y,2026-02-20
1,2026-02-11,CW00000,CW00000,L,NX,AAZH26L-AWXLBA,VF,223-43,D4,Y,Y,2026-02-20
2,2026-02-10,JM00000,JM00000,T,APD,AAHH45L-PFXLBK,VD,089-20,D4,Y,Y,2026-02-10
3,2026-02-08,TD00000,TD10102,T,SIE,AXLH45L-PNXLHA,SX,1G3-25,D4,Y,Y,2026-02-08
4,2026-02-07,HS00000,HS10101,T,RAV,AXAH52L-ANXMBA,5W,089-20,D4,Y,Y,2026-02-07
5,2026-02-07,HS00000,HS00000,T,APD,AAHH45L-PFXLBK,VD,1L5-20,D4,Y,Y,2026-02-07
6,2026-02-07,CT00000,CT30104,L,NX,AAZH25L-AWXLBA,VC,083-40,D4,Y,Y,2026-02-11
7,2026-02-07,KJ00000,KJ00000,T,RAV,AXAH52L-ANXMBA,5W,089-20,D4,Y,Y,2026-02-07
8,2026-02-07,LS00000,LS10102,T,RAV,AXAH52L-ANXMBA,5W,089-20,D4,Y,Y,2026-02-07
9,2026-02-06,DT00000,DT30107,L,NX,AAZH25L-AWXLBA,VB,223-31,D4,Y,Y,2026-02-06


In [3]:
# 딜러 수 / 모델 수 / 계약일 범위 — 실측 근거(ML 설계서 상 딜러 21·모델 39와 비교)
pd.read_sql("""
SELECT
    COUNT(DISTINCT dealer_id) AS dealer_cnt,
    COUNT(DISTINCT model_cd)  AS model_cnt,
    MIN(contract_dt) AS min_contract_dt,
    MAX(contract_dt) AS max_contract_dt,
    COUNT(*) AS row_cnt
FROM dbo.OM_CONTRACT
""", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2832888898.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,dealer_cnt,model_cnt,min_contract_dt,max_contract_dt,row_cnt
0,20,35,2001-01-02,2026-02-12,410997


## 2. 딜러·모델별 일별/월별 타겟 — `ktws.FCT_SALES_TARGET_DAILY`

`ac_trgt`(월 누적 타겟), `co_trgt`, `ac_trgt_daily`/`co_trgt_daily`(일별 배분 타겟)를 확인한다.

In [4]:
sales_target = pd.read_sql("""
SELECT TOP 1000
    dealer_key,
    brand_cd,
    model_key,
    mon_target_variant,
    dateby,
    ac_trgt,
    co_trgt,
    ac_trgt_daily,
    co_trgt_daily
FROM ktws.FCT_SALES_TARGET_DAILY
ORDER BY dateby DESC
""", conn)

print(sales_target.shape)
sales_target.head(20)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\1650717149.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sales_target = pd.read_sql("""


(1000, 9)


,dealer_key,brand_cd,model_key,mon_target_variant,dateby,ac_trgt,co_trgt,ac_trgt_daily,co_trgt_daily
0,LEXUSCT00000,L,LEXUSRX,RX450h+,2026-06-30,4.0,15.0,0.0,0.0
1,LEXUSJB00000,L,LEXUSNX,NX350h,2026-06-30,17.0,23.0,1.0,1.0
2,TOYOTATD00000,T,TOYOTASIE,SIENNA-HV AWD,2026-06-30,7.0,8.0,0.0,0.0
3,LEXUSDT00000,L,LEXUSRX,RX450h+,2026-06-30,8.0,28.0,1.0,1.0
4,LEXUSNY00000,L,LEXUSRX,RX350h(N),2026-06-30,10.0,16.0,1.0,1.0
5,LEXUSKM00000,L,LEXUSRX,RX450h+,2026-06-30,3.0,11.0,0.0,0.0
6,TOYOTAKJ00000,T,TOYOTAPRI,PRIUS PHEV,2026-06-30,3.0,5.0,0.0,0.0
7,LEXUSYM00000,L,LEXUSRX,RX350h(N),2026-06-30,19.0,29.0,1.0,1.0
8,LEXUSSY00000,L,LEXUSRX,RX450h+,2026-06-30,2.0,8.0,0.0,0.0
9,LEXUSCT00000,L,LEXUSNX,NX350h,2026-06-30,50.0,69.0,4.0,2.0


In [5]:
# 타겟 결측률 확인 (ML 설계서: ac_trgt 약 28% 결측으로 보고됨)
pd.read_sql("""
SELECT
    COUNT(*) AS row_cnt,
    SUM(CASE WHEN ac_trgt IS NULL THEN 1 ELSE 0 END) AS ac_trgt_null_cnt,
    MIN(dateby) AS min_dateby,
    MAX(dateby) AS max_dateby
FROM ktws.FCT_SALES_TARGET_DAILY
""", conn)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\3133359077.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql("""


,row_cnt,ac_trgt_null_cnt,min_dateby,max_dateby
0,91089,25392,2025-01-01,2026-06-30


## 3. 차원 테이블 — `dbo.VS_SFX`, `dbo.VS_MODEL`, `ktws.DIM_CALENDAR`

In [6]:
vs_model = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_MODEL", conn)
vs_sfx = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_SFX", conn)
dim_calendar = pd.read_sql("SELECT TOP 200 * FROM ktws.DIM_CALENDAR ORDER BY 1 DESC", conn)

print("VS_MODEL:", vs_model.shape)
print("VS_SFX:", vs_sfx.shape)
print("DIM_CALENDAR:", dim_calendar.shape)
vs_model.head()

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2240321022.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vs_model = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_MODEL", conn)
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2240321022.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  vs_sfx = pd.read_sql("SELECT TOP 200 * FROM dbo.VS_SFX", conn)
C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\2240321022.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dim_calendar = pd.read_sql("SELECT T

VS_MODEL: (135, 11)
VS_SFX: (200, 41)
DIM_CALENDAR: (200, 16)


,brand_cd,model_cd,model_nm,curr_sales_yn,display_order,reg_dt,upd_dt,reg_user_id,upd_user_id,ELT_TIMESTAMP,BRAND
0,L,LS,LS,Y,1.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,2222222222222,2026-07-03T13:12:08.0867473Z,LEXUS
1,L,GS,GS,Y,5.0,2011-01-02 02:33:16.0000000,2019-11-27 10:06:56.0000000,ADMIN,TMR040161,2026-07-03T13:12:08.0867473Z,LEXUS
2,L,ES,ES,Y,10.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,ADMIN,2026-07-03T13:12:08.0867473Z,LEXUS
3,L,IS,IS,Y,15.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,ADMIN,2026-07-03T13:12:08.0867473Z,LEXUS
4,L,RX,RX,Y,20.0,2011-01-02 02:33:16.0000000,2011-01-02 02:33:16.0000000,ADMIN,2222222222222,2026-07-03T13:12:08.0867473Z,LEXUS


In [7]:
vs_sfx.head()

,brand_cd,model_cd,variant_cd,my_cd,sfx_cd,brand_nm,model_nm,variant_nm,model_year,sfx_nm,curr_sales_yn,display_order,launch_dt,form_apply,motive_type,taking_fix,displacement,hp,gross_weight,cylinder_cnt,max_load,max_output,length,width,height,order_yn,reg_dt,upd_dt,hybrid_yn,navi_yn,ecfrnd_vhcle_knd,grade,connected_car_yn,spec_variant_nm,hi_pass_yn,black_box_yn,consign_sales_flag,ew_yn,dcm_type,ELT_TIMESTAMP,BRAND
0,L,CT,ZWA10L-AHXBBA,2020,CY,LEXUS,CT,CT200h,2020,CY,N,2.0,20190424,010-2-00034-0012-1219,2ZR,5,1798,99,1780,4.0,0,136.0,4355.0,1765.0,1455.0,N,2019-02-25 18:42:34.0000000,2023-08-11 10:14:43.0000000,Y,N,2,Supreme,None,렉서스 CT200h,None,None,N,N,None,2026-07-03T13:13:02.0023861Z,LEXUS
1,T,T86,ZN6-FYE7,2018,8H,Toyota,TOYOTA 86,86 AT,2018,8H,N,11.0,20180201,010-2-00042-0007-1217,FA20,4,1998,203,1540,4.0,0,203.0,4240.0,1775.0,1320.0,N,2017-09-29 19:40:23.0000000,2023-03-03 10:30:38.0000000,N,N,NaN,AT,None,토요타 86 AT,None,None,N,N,None,2026-07-03T13:11:19.234407Z,TOYOTA
2,T,T86,ZN6-GYE8,2019,8M,Toyota,TOYOTA 86,86 MT,2019,8M,N,17.0,20180701,010-2-00043-0008-1218,FA20,4,1998,203,1500,4.0,0,203.0,4240.0,1775.0,1320.0,N,2018-05-03 13:58:14.0000000,2023-03-03 10:30:38.0000000,N,N,NaN,MT,None,토요타 86 MT,None,None,N,N,None,2026-07-03T13:11:19.234407Z,TOYOTA
3,L,IS,GXE10L-AEPVKW,2000,IS,LEXUS,IS,IS200,2000,IS,N,1.0,20020801,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N,2006-03-15 00:00:00.0000000,2017-05-23 17:18:42.0000000,N,N,NaN,NaN,None,렉서스 IS200,None,None,NaN,N,None,2026-07-03T13:13:02.0023861Z,LEXUS
4,L,GS,UZS190L-BETQKA,2008,GP,LEXUS,GS,GS430,2008,GP,N,0.0,20071101,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,N,2007-06-11 09:36:00.0000000,2017-05-23 17:18:42.0000000,N,N,NaN,NaN,None,렉서스 GS430,None,None,NaN,N,None,2026-07-03T13:13:02.0023861Z,LEXUS


In [8]:
dim_calendar.head()

,Date,Year,QuarterNumber,MonthNumber,Day,DayName,WeekNumber,WeekNumber_Monthly,WeekNumber_Monthly_txt,QuarterText,MonthName,MonthAbbr,WeekdayName,WeekdayAbbr,YearMonth,WeekdayNumber
0,2026-08-31,2026,3,8,31,31일,36,4,4주차,Q3,8월,8월,월요일,월,2026-08,1
1,2026-08-30,2026,3,8,30,30일,36,4,4주차,Q3,8월,8월,일요일,일,2026-08,7
2,2026-08-29,2026,3,8,29,29일,35,4,4주차,Q3,8월,8월,토요일,토,2026-08,6
3,2026-08-28,2026,3,8,28,28일,35,4,4주차,Q3,8월,8월,금요일,금,2026-08,5
4,2026-08-27,2026,3,8,27,27일,35,4,4주차,Q3,8월,8월,목요일,목,2026-08,4


## 4. 딜러×월 실적 vs 타겟 결합 미리보기

`OM_CONTRACT`를 딜러×월로 집계한 실적과 `FCT_SALES_TARGET_DAILY`의 월 타겟(`ac_trgt`)을 붙여 달성률을 미리 확인한다.

> 주의: `contract_dt`가 문자열(varchar)이므로 정제 후 사용. `eod_month=44652` 등 더티 날짜값은 제외해야 한다.

In [9]:
dealer_month_actual = pd.read_sql("""
SELECT
    dealer_id,
    LEFT(contract_dt, 6) AS yyyymm,
    COUNT(*) AS contract_cnt,
    SUM(CASE WHEN sold_yn = 'Y' THEN 1 ELSE 0 END) AS sold_cnt
FROM dbo.OM_CONTRACT
WHERE contract_dt LIKE '2[0-9][0-9][0-9]%'  -- 더티 날짜값 제외
GROUP BY dealer_id, LEFT(contract_dt, 6)
ORDER BY yyyymm DESC, dealer_id
""", conn)

print(dealer_month_actual.shape)
dealer_month_actual.head(20)

(755, 4)


C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\3321529729.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dealer_month_actual = pd.read_sql("""


,dealer_id,yyyymm,contract_cnt,sold_cnt
0,CT00000,2026-0,12,8
1,CW00000,2026-0,22,21
2,DM00000,2026-0,14,11
3,DT00000,2026-0,11,8
4,HS00000,2026-0,18,13
5,JB00000,2026-0,3,3
6,JM00000,2026-0,7,7
7,KJ00000,2026-0,17,16
8,KM00000,2026-0,9,7
9,LS00000,2026-0,23,19


In [10]:
dealer_month_target = pd.read_sql("""
SELECT
    dealer_key,
    FORMAT(dateby, 'yyyyMM') AS yyyymm,
    SUM(ac_trgt) AS ac_trgt_sum
FROM ktws.FCT_SALES_TARGET_DAILY
GROUP BY dealer_key, FORMAT(dateby, 'yyyyMM')
ORDER BY yyyymm DESC, dealer_key
""", conn)

print(dealer_month_target.shape)
dealer_month_target.head(20)

C:\Users\LouisLee(이시호)\AppData\Local\Temp\ipykernel_31904\448800596.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  dealer_month_target = pd.read_sql("""


(291, 3)


,dealer_key,yyyymm,ac_trgt_sum
0,LEXUSCT00000,202606,2534.0
1,LEXUSCW00000,202606,2935.0
2,LEXUSDT00000,202606,4594.0
3,LEXUSJB00000,202606,854.0
4,LEXUSKM00000,202606,1855.0
5,LEXUSNY00000,202606,940.0
6,LEXUSSY00000,202606,1266.0
7,LEXUSYM00000,202606,1802.0
8,TOYOTADM00000,202606,1389.0
9,TOYOTAHS00000,202606,2082.0


In [11]:
achievement_preview = dealer_month_actual.merge(
    dealer_month_target,
    left_on=['dealer_id', 'yyyymm'],
    right_on=['dealer_key', 'yyyymm'],
    how='left'
)
achievement_preview['achieve_rate'] = (
    achievement_preview['sold_cnt'] / achievement_preview['ac_trgt_sum']
)
achievement_preview.sort_values('yyyymm', ascending=False).head(30)

,dealer_id,yyyymm,contract_cnt,sold_cnt,dealer_key,ac_trgt_sum,achieve_rate
0,CT00000,2026-0,12,8,NaN,NaN,NaN
1,CW00000,2026-0,22,21,NaN,NaN,NaN
2,DM00000,2026-0,14,11,NaN,NaN,NaN
3,DT00000,2026-0,11,8,NaN,NaN,NaN
4,HS00000,2026-0,18,13,NaN,NaN,NaN
5,JB00000,2026-0,3,3,NaN,NaN,NaN
6,JM00000,2026-0,7,7,NaN,NaN,NaN
7,KJ00000,2026-0,17,16,NaN,NaN,NaN
8,KM00000,2026-0,9,7,NaN,NaN,NaN
9,LS00000,2026-0,23,19,NaN,NaN,NaN
